# 00 · Environment & smoke test
Goal: prove the harness works **before** burning GPU hours. Order: (1) install, (2) smoke tier on synthetic data (CPU, seconds),
(3) per-model sanity check on your GPU, (4) get TIME data.  Works on a local GPU box, Kaggle, or Colab.

In [ ]:
import os, sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, yaml
pd.set_option("display.width", 160, "display.max_columns", 30)

## 1 · Install (run once; skip cells you already did)

In [ ]:
# Core harness (CPU is enough for this one)
# !pip install -e .
# Model stacks -- install only what you will run. Pin versions in requirements-models.txt once everything works.
# !pip install -r requirements-models.txt
# TIME benchmark code + data (HF dataset Real-TSF/TIME, CC BY-NC 4.0 -> fine for academic use)
# !git clone https://github.com/zqiao11/TIME ../TIME && pip install -e ../TIME
# from huggingface_hub import snapshot_download
# snapshot_download("Real-TSF/TIME", repo_type="dataset", local_dir="data/TIME-dataset")   # large: check disk first
# os.environ["TIME_STORAGE"] = str(ROOT / "data/TIME-dataset")

In [ ]:
try:
    import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except ImportError:
    print("torch not installed -> only baselines + analysis will run")

## 2 · Smoke tier (synthetic tasks, statistical baselines) — must pass on any machine

In [ ]:
from tscompare import runner, aggregate as ag
from tscompare.models.registry import build
from tscompare.tasks import make_synthetic_task
tasks = [make_synthetic_task(name=f"synthetic/{i}", seed=i, n_series=12) for i in range(6)]
for m in ["seasonal_naive", "naive", "seasonal_avg", "drift"]:
    runner.run(build(m), tasks, out_dir="/tmp/smoke_nb", verbose=False, overwrite=True)
df = ag.load_results("/tmp/smoke_nb")
ag.summary_table(df)[["family", "mase_rel", "wql_rel", "skill_%", "avg_rank", "win_rate_%", "tasks_ok"]].round(3)

Expected: `seasonal_naive` has `mase_rel == 1.000` and ranks first on this seasonal synthetic data. If not, stop and debug here.

## 3 · Per-model sanity check on the GPU
Runs each wrapper on a tiny synthetic task. A failure here is almost always **API drift in the upstream library** → fix the single wrapper
class (see `docs/MODEL_NOTES.md`). Don't proceed until every method you intend to report says `OK`.

In [ ]:
!python scripts/check_models.py          # all methods; or e.g.: python scripts/check_models.py sundial timesfm

## 4 · Done when
- [ ] smoke tier table above looks right  - [ ] `check_models.py` is all `OK` for the methods you will report  - [ ] `TIME_STORAGE` points to the dataset
Next → `01_data_and_protocol.ipynb`.